In [15]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [16]:
load_dotenv()
SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')

## Tool Callings

In [17]:
from sarvamai import SarvamAI
import json

In [18]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

In [19]:
system_prompt = """
You are a helpful Customer Services Rep working for ICICI Bank.
"""

In [20]:
model="sarvam-105b-conversations",
messages=[
    {"role": "system", "content": system_prompt},
]

In [21]:
def get_balance(account_number: str):
    if account_number == '001002':
        return 'INR 51203'
    else:
        return 'INR 2500'

def get_customer_id(account_number: str):
    return 'A12378'

In [22]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
     {
        "type": "function",
        "function": {
            "name": "get_customer_id",
            "description": "Get the unique id identifying the customer",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },   
]

In [23]:
tools_map  = {
    'get_balance': get_balance,
    'get_customer_id': get_customer_id,
}

def invoke_tool(f_name: str, f_args: dict):
    return tools_map[f_name](**f_args)

In [36]:
# invoke_tool('get_balance', {'account_number': '001002'})

In [24]:
# print("\033[91m This is Red Text \033[0m")
# print("\033[92m This is Green Text \033[0m")
# print("\033[94m This is Blue Text \033[0m")
GREEN_TEXT = "\033[92m"
BLUE_TEXT = "\033[94m"
BLACK_TEXT = "\033[0m"

In [29]:
user_message = ''
while True:
    user_message = input('\nUser:')
    if user_message == 'quit':
        break
    messages.append(
        {"role": "user", "content": user_message.strip()}
    )

    while True:
        response = client.chat.completions(
            model="sarvam-105b-conversations",
            messages=messages,
            tools=tools,
        )
    
        # print(response.choices[0].message)
        if response.choices[0].message.tool_calls:
            print('AI made decision to select a tool')
            message = response.choices[0].message
            for tool_call in message.tool_calls:
                f_name = tool_call.function.name
                f_args = json.loads(tool_call.function.arguments)
                result = invoke_tool(f_name, f_args)
    
                messages.append(
                    {
                        "role": "assistant",
                        "tool_calls": [
                            {
                                "id": tool_call.id,
                                "type": "function",
                                "function": {
                                    "name": tool_call.function.name,
                                    "arguments": tool_call.function.arguments,
                                },
                            }
                        ],
                    }
                )
    
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result,
                    }
                )
            continue
        break

    print(f'\n{BLUE_TEXT}Assistant: ', response.choices[0].message.content.strip(), BLACK_TEXT)
    messages.append(
        {"role": "assistant", "content": response.choices[0].message.content}
    )


User: hi



Assistant:  Hi! No worries at all. Whenever you're ready, just share what you need help with—whether it's checking your account balance, looking up your customer ID, or anything else. I'm here for you!



User: how are you ?



Assistant:  I'm doing well, thank you for asking! I'm here to make sure your banking needs are taken care of. How about you—how can I help you today?



User: balance for 001002


AI made decision to select a tool

Assistant:  The current balance for account number **001002** is **₹51,203**. Is there anything else I can help you with?



User: thanks



Assistant:  You're welcome! If you need any further assistance, feel free to ask. Have a great day!



User: quit
